In [0]:
from pyspark.sql.functions import to_date, col, row_number, lit, expr, regexp_replace, upper
from pyspark.sql.window import Window

bronze_schemapath = "subscription_pipeline.bronze"
silver_schemapath = "subscription_pipeline.silver"

In [0]:
# Remove Fivetran ingested columns
fivetran_ingested_column_to_remove = [
    "_file",
    "_fivetran_synced",
    "_modified",
    "_line"
]

def remove_fivetran_ingested_column(df):
    for column in fivetran_ingested_column_to_remove:
        df = df.drop(column)
    return df

In [0]:
# Read from bronze and remove metadata columns
df = spark.read.table(f"{bronze_schemapath}.opportunity")
df_clean = remove_fivetran_ingested_column(df)

# Convert date columns
df_clean = df_clean.withColumn(
    "start_date",
    to_date(col("start_date"), "dd-MM-yyyy")
)
df_clean = df_clean.withColumn(
    "end_date",
    to_date(col("end_date"), "dd-MM-yyyy")
)

# Remove duplicates
df_clean = df_clean.dropDuplicates()

# Fill missing opportunity_id if necessary, starting with 500000
if "opportunity_id" not in df_clean.columns or df_clean.filter(col("opportunity_id").isNull()).count() > 0:
    w = Window().orderBy(lit(1))
    df_clean = df_clean.withColumn("opportunity_id", row_number().over(w) + 499999)

# Clean revenue_amount: remove currency symbols and ensure numeric type
df_clean = df_clean.withColumn(
    "revenue_amount",
    regexp_replace(col("revenue_amount"), '[$£]', '')
)
df_clean = df_clean.withColumn(
    "revenue_amount",
    regexp_replace(col("revenue_amount"), '[^0-9\\.-]', '')
)
df_clean = df_clean.withColumn(
    "revenue_amount",
    col("revenue_amount").cast("double")
)

# Clean close_status: standardize to uppercase
df_clean = df_clean.withColumn("close_status", upper(col("close_status")))

# Write to silver
df_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schemapath}.opportunity")

print(f"✓ Opportunity table created: {df_clean.count()} rows")